# Data Discovery & Validation
## Singapore Jobs Analytics

**Objective**: Load data, validate constraints, perform statistical profiling, and run critical validations for project planning.

**Critical Validations**:
1. Temporal coverage (determines forecasting approach)
2. File size (determines GitHub deployment strategy)
3. JSON parsing (categories field structure)

**Outputs**:
- Schema and statistics
- Distribution analysis
- Validation report with go/no-go decisions

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries loaded successfully")

## 1. Data Loading with Optimization

In [ ]:
# Load data with dtype optimization
data_path = Path('../SGJobData.csv')

print(f"Loading data from: {data_path}")
print(f"File size: {data_path.stat().st_size / (1024**2):.1f} MB\n")

# Initial load to inspect columns
df_sample = pd.read_csv(data_path, nrows=1000)
print("Sample columns:")
print(df_sample.columns.tolist())
print(f"\nSample shape: {df_sample.shape}")

In [ ]:
# Load full dataset with optimized dtypes
dtype_spec = {
    'categories': 'object',  # JSON string
    'company': 'category',
    'employmentType': 'category',
    'jobTitle': 'object',  # High cardinality
    'positionLevel': 'category'
}

print("Loading full dataset...")
df = pd.read_csv(data_path, dtype=dtype_spec, low_memory=False)

print(f"\n✅ Data loaded successfully")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

## 2. Initial Schema Inspection

In [ ]:
# Display basic info
print("Dataset Info:")
print("=" * 80)
df.info()

In [ ]:
# First few rows
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Column overview
print("\nColumn Overview:")
print("=" * 80)

col_info = pd.DataFrame({
    'Column': df.columns,
    'Non-Null Count': [df[col].notna().sum() for col in df.columns],
    'Null Count': [df[col].isna().sum() for col in df.columns],
    'Null %': [f"{df[col].isna().sum() / len(df) * 100:.2f}%" for col in df.columns],
    'Dtype': [str(df[col].dtype) for col in df.columns],
    'Unique': [df[col].nunique() for col in df.columns],
    'Sample Value': [str(df[col].dropna().iloc[0])[:50] if df[col].notna().any() else 'N/A' for col in df.columns]
})

col_info

## 3. CRITICAL VALIDATION #1: Temporal Coverage Check

**Purpose**: Determine if dataset has sufficient time span for forecasting models.

**Decision Criteria**:
- < 6 months: Use trend analysis only (no forecasting)
- 6-12 months: Basic trend decomposition, avoid Prophet
- > 12 months: Prophet/ARIMA feasible

In [ ]:
# Check for date columns
date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
print(f"Date-related columns found: {date_cols}\n")

# Parse date columns
for col in date_cols:
    try:
        df[f'{col}_parsed'] = pd.to_datetime(df[col], errors='coerce')
        print(f"✅ Successfully parsed: {col}")
    except Exception as e:
        print(f"❌ Failed to parse {col}: {e}")

In [ ]:
# Temporal coverage analysis
print("\n" + "="*80)
print("TEMPORAL COVERAGE VALIDATION")
print("="*80 + "\n")

for col in [c for c in df.columns if c.endswith('_parsed')]:
    date_range = df[col].agg(['min', 'max', 'count'])
    non_null = df[col].notna().sum()
    
    if non_null > 0:
        time_span_days = (date_range['max'] - date_range['min']).days
        time_span_months = time_span_days / 30
        
        print(f"Column: {col}")
        print(f"  Date range: {date_range['min'].date()} to {date_range['max'].date()}")
        print(f"  Time span: {time_span_days} days ({time_span_months:.1f} months)")
        print(f"  Non-null records: {non_null:,} ({non_null/len(df)*100:.1f}%)\n")
        
        # Decision logic
        if time_span_months < 6:
            print("  ⚠️ DECISION: INSUFFICIENT DATA FOR FORECASTING")
            print("  → Use trend analysis and descriptive statistics only")
            print("  → Skip Prophet/ARIMA models\n")
            forecasting_feasible = False
        elif time_span_months < 12:
            print("  ⚠️ DECISION: LIMITED DATA")
            print("  → Use basic trend decomposition (moving averages, seasonal patterns)")
            print("  → Avoid complex models like Prophet\n")
            forecasting_feasible = 'limited'
        else:
            print("  ✅ DECISION: SUFFICIENT DATA FOR FORECASTING")
            print("  → Prophet/ARIMA models are feasible")
            print("  → Can generate 3-6 month predictions\n")
            forecasting_feasible = True

## 4. CRITICAL VALIDATION #2: File Size Check

**Purpose**: Determine deployment strategy based on processed data size.

**Decision Criteria**:
- Parquet < 50 MB: Can push Silver layer to GitHub
- Parquet > 50 MB: Use Gold layer only (pre-aggregated)

In [ ]:
# Test Parquet compression
print("="*80)
print("FILE SIZE VALIDATION")
print("="*80 + "\n")

test_parquet_path = Path('../data/silver/test_size.parquet')
test_parquet_path.parent.mkdir(parents=True, exist_ok=True)

print("Testing Parquet compression...")
df.to_parquet(test_parquet_path, compression='snappy', index=False)

size_mb = test_parquet_path.stat().st_size / (1024**2)
compression_ratio = (data_path.stat().st_size / (1024**2)) / size_mb

print(f"\nOriginal CSV size: {data_path.stat().st_size / (1024**2):.1f} MB")
print(f"Parquet size (snappy): {size_mb:.1f} MB")
print(f"Compression ratio: {compression_ratio:.1f}x\n")

# Decision logic
if size_mb > 50:
    print(f"⚠️ DECISION: FILE TOO LARGE FOR GITHUB ({size_mb:.1f}MB)")
    print("→ Dashboard will load from Gold layer only (pre-aggregated data)")
    print("→ Silver layer excluded from git (add to .gitignore)")
    print("→ Estimated Gold layer size: ~5-10MB (aggregated metrics)\n")
    github_compatible = False
else:
    print(f"✅ DECISION: GITHUB COMPATIBLE ({size_mb:.1f}MB)")
    print("→ Can include Silver layer in repository")
    print("→ Dashboard can load from Silver layer for full flexibility\n")
    github_compatible = True

# Clean up test file
test_parquet_path.unlink()
print("Test file removed.")

## 5. CRITICAL VALIDATION #3: JSON Category Parsing

**Purpose**: Understand category structure and decide on handling strategy.

**Questions**:
- Are categories single or multiple per job?
- What's the structure of the JSON?
- Should we use primary category or all categories?

In [ ]:
print("="*80)
print("CATEGORY STRUCTURE VALIDATION")
print("="*80 + "\n")

# Sample category values
print("Sample category values:")
print(df['categories'].head(3).tolist())
print()

In [ ]:
# Parse categories JSON
def safe_json_parse(x):
    try:
        return json.loads(x) if pd.notna(x) else []
    except:
        return []

df['categories_parsed'] = df['categories'].apply(safe_json_parse)
df['num_categories'] = df['categories_parsed'].apply(len)

print("Category multiplicity analysis:")
print(df['num_categories'].value_counts().sort_index())
print(f"\nJobs with multiple categories: {(df['num_categories'] > 1).sum():,} ({(df['num_categories'] > 1).mean()*100:.1f}%)")
print(f"Average categories per job: {df['num_categories'].mean():.2f}")

In [ ]:
# Extract primary category (first in list)
def extract_primary_category(cat_list):
    if cat_list and len(cat_list) > 0:
        return cat_list[0].get('category', 'Unknown')
    return 'Unknown'

df['primary_category'] = df['categories_parsed'].apply(extract_primary_category)

print("\nPrimary category distribution (top 20):")
primary_cat_counts = df['primary_category'].value_counts().head(20)
print(primary_cat_counts)
print(f"\nTotal unique primary categories: {df['primary_category'].nunique()}")

In [ ]:
# Decision summary
print("\n" + "="*80)
print("✅ DECISION: CATEGORY HANDLING STRATEGY")
print("="*80)
print("→ Use PRIMARY CATEGORY (first in list) for main aggregations")
print("→ Avoids inflated job counts from multi-category jobs")
print("→ Store full category list as secondary attribute for drill-down")
print(f"→ Working with {df['primary_category'].nunique()} distinct categories\n")

## 6. Statistical Profiling

In [ ]:
# Numeric columns summary
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}\n")

if numeric_cols:
    print("Numeric statistics:")
    display(df[numeric_cols].describe())

In [ ]:
# Categorical columns summary
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"\nCategorical columns ({len(categorical_cols)}):")

cat_summary = pd.DataFrame({
    'Column': categorical_cols,
    'Unique Values': [df[col].nunique() for col in categorical_cols],
    'Most Common': [df[col].mode()[0] if len(df[col].mode()) > 0 else 'N/A' for col in categorical_cols],
    'Most Common Count': [df[col].value_counts().iloc[0] if df[col].notna().any() else 0 for col in categorical_cols],
    'Most Common %': [f"{df[col].value_counts(normalize=True).iloc[0]*100:.1f}%" if df[col].notna().any() else '0%' for col in categorical_cols]
})

cat_summary

## 7. Key Field Analysis

In [ ]:
# Salary analysis (if exists)
salary_cols = [col for col in df.columns if 'salary' in col.lower()]

if salary_cols:
    print(f"Salary columns found: {salary_cols}\n")
    
    for col in salary_cols:
        if df[col].dtype in [np.number, 'int64', 'float64']:
            print(f"\n{col}:")
            print(f"  Non-null: {df[col].notna().sum():,} ({df[col].notna().sum()/len(df)*100:.1f}%)")
            print(f"  Range: ${df[col].min():,.0f} - ${df[col].max():,.0f}")
            print(f"  Median: ${df[col].median():,.0f}")
            print(f"  Mean: ${df[col].mean():,.0f}")
            print(f"  Std: ${df[col].std():,.0f}")

In [ ]:
# Position level distribution
if 'positionLevel' in df.columns:
    print("\nPosition Level Distribution:")
    print(df['positionLevel'].value_counts())
    
    # Salary by position level
    if salary_cols:
        salary_col = salary_cols[0]
        print(f"\nMedian {salary_col} by Position Level:")
        print(df.groupby('positionLevel')[salary_col].median().sort_values(ascending=False))

In [ ]:
# Employment type distribution
if 'employmentType' in df.columns:
    print("\nEmployment Type Distribution:")
    emp_type_dist = df['employmentType'].value_counts()
    print(emp_type_dist)
    print(f"\nTotal unique employment types: {df['employmentType'].nunique()}")

In [ ]:
# Top companies by job postings
if 'company' in df.columns:
    print("\nTop 20 Companies by Job Postings:")
    print(df['company'].value_counts().head(20))

## 8. Distribution Visualizations

In [ ]:
# Salary distribution
if salary_cols:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    salary_col = salary_cols[0]
    salary_data = df[salary_col].dropna()
    
    # Histogram
    axes[0].hist(salary_data, bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Salary ($)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title(f'{salary_col} Distribution')
    axes[0].grid(True, alpha=0.3)
    
    # Box plot
    axes[1].boxplot(salary_data, vert=True)
    axes[1].set_ylabel('Salary ($)')
    axes[1].set_title(f'{salary_col} Box Plot')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Category distribution
if 'primary_category' in df.columns:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    top_categories = df['primary_category'].value_counts().head(20)
    top_categories.plot(kind='barh', ax=ax, color='steelblue')
    
    ax.set_xlabel('Number of Job Postings')
    ax.set_ylabel('Category')
    ax.set_title('Top 20 Job Categories by Posting Volume')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Temporal distribution (if date available)
date_parsed_cols = [c for c in df.columns if c.endswith('_parsed')]

if date_parsed_cols:
    date_col = date_parsed_cols[0]
    
    fig, ax = plt.subplots(figsize=(15, 5))
    
    # Group by month
    df_with_date = df[df[date_col].notna()].copy()
    df_with_date['year_month'] = df_with_date[date_col].dt.to_period('M')
    monthly_counts = df_with_date.groupby('year_month').size()
    
    monthly_counts.plot(ax=ax, linewidth=2, marker='o')
    ax.set_xlabel('Month')
    ax.set_ylabel('Number of Job Postings')
    ax.set_title('Job Postings Over Time (Monthly)')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 9. Validation Report Summary

In [ ]:
# Compile validation report
validation_report = {
    'dataset_info': {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'memory_usage_mb': df.memory_usage(deep=True).sum() / (1024**2),
        'file_size_mb': data_path.stat().st_size / (1024**2)
    },
    'temporal_coverage': {
        'date_columns': date_cols,
        'forecasting_feasible': forecasting_feasible if 'forecasting_feasible' in locals() else 'unknown',
        'date_range': {
            'min': str(df[date_parsed_cols[0]].min()) if date_parsed_cols else None,
            'max': str(df[date_parsed_cols[0]].max()) if date_parsed_cols else None
        } if date_parsed_cols else None
    },
    'file_size_check': {
        'parquet_size_mb': size_mb if 'size_mb' in locals() else None,
        'github_compatible': github_compatible if 'github_compatible' in locals() else None,
        'deployment_strategy': 'Gold layer only' if not github_compatible else 'Silver layer included'
    },
    'category_structure': {
        'total_unique_categories': df['primary_category'].nunique() if 'primary_category' in df.columns else None,
        'multi_category_jobs_pct': (df['num_categories'] > 1).mean() * 100 if 'num_categories' in df.columns else None,
        'strategy': 'Use primary category for aggregations'
    },
    'data_quality_flags': {
        'missing_values': df.isnull().sum().to_dict(),
        'zero_variance_cols': [col for col in df.columns if df[col].nunique() == 1]
    }
}

# Save validation report
report_path = Path('../data/silver/validation_report.json')
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(validation_report, f, indent=2, default=str)

print("="*80)
print("VALIDATION REPORT SUMMARY")
print("="*80)
print(json.dumps(validation_report, indent=2, default=str))
print(f"\n✅ Report saved to: {report_path}")

## 10. Next Steps

Based on validation results:

1. **Proceed to Data Quality Assessment** (02_data_quality.ipynb)
2. **Forecasting Approach**: Check `forecasting_feasible` flag
3. **Deployment Strategy**: Check `github_compatible` flag
4. **Category Handling**: Use `primary_category` for main analysis

In [ ]:
# Save processed sample for next notebook
sample_df = df.head(10000).copy()
sample_path = Path('../data/silver/sample_data.parquet')
sample_df.to_parquet(sample_path, compression='snappy', index=False)
print(f"Sample data saved to: {sample_path}")
print(f"Sample size: {len(sample_df):,} rows")